# 02. Zewnętrzne API (Open-Meteo) pogody

Celem tego notatnika jest zautomatyzowane pobranie historycznych danych pogodowych przy użyciu publicznego API Open-Meteo wykorzystując współrzędne urządzeń z `devices.csv`. Dane celowo utrzymane są w swojej natywnej gęstości API, czyli spływając w odstępach godzinowych **(1H)**.

Zgodnie z wymaganiami zakres czasu sztywno zaplanowany jest w granicach: **Październik 2024 r. (Oct 2024) - Kwiecień 2025 r. (Apr 2025)**.

Zapisane zestawienie trafia finalnie do pliku `weather.csv`.

In [1]:
import pandas as pd
import requests
import time
from tqdm.auto import tqdm
import datetime
import warnings

warnings.filterwarnings('ignore')

## 1. Wyznaczenie horyzontu czasowego

Zdefiniowany obustronnie brzeg.

In [2]:
DEVICES_PATH = 'data/devices.csv'

start_date = '2024-10-01'
end_date = '2025-04-30'

print(f"Horyzont pobierania API: Od {start_date} do {end_date}")

Horyzont pobierania API: Od 2024-10-01 do 2025-04-30


## 2. Pętla wykonawcza z API Open-Meteo

Baza stacji i lokalizacji jest przeczesywana. Pytamy w endpoint (np. darmowy serwer Archive API) o popularne mierniki pogodowe.

In [3]:
devices_df = pd.read_csv(DEVICES_PATH)
print(f"Wczytano urządzeń z bazy: {len(devices_df)}")

def fetch_weather(lat, lon, st, ed):
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    payload = {
        "latitude": lat,
        "longitude": lon,
        "start_date": st,
        "end_date": ed,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,cloud_cover",
        "timezone": "UTC"
    }
    
    resp = requests.get(url, params=payload)
    
    if resp.status_code == 400 and "not available" in str(resp.content):
        # Jeżeli darmowe Archive API zablokuje np. bieżący styczeń czy luty (najsilniejsze ramy < 5dni starych instancji), 
        # opieramy na fallback'u forecastowym trzymającym pomiary bieżącego roku (-90 dni wstecz i teraźniejszość)
        url_recent = "https://api.open-meteo.com/v1/forecast"
        payload.pop("start_date", None)
        payload.pop("end_date", None)
        payload["past_days"] = 92 
        resp = requests.get(url_recent, params=payload)
        
    if resp.status_code == 200:
        return resp.json()
    else:
        return None

Wczytano urządzeń z bazy: 600


In [4]:
all_weather_dfs = []

for idx, row in tqdm(devices_df.iterrows(), total=len(devices_df), desc="Odpytywanie Punktów Open-Meteo"):
    lat, lon = row['latitude'], row['longitude']
    dev_id = row['deviceId']
    
    data_json = fetch_weather(lat, lon, start_date, end_date)
    
    if data_json and 'hourly' in data_json:
        # Tłumaczenie do prostej ramki Pandas i zaszczepianie sygnatury stacji 
        df_tmp = pd.DataFrame(data_json['hourly'])
        df_tmp['deviceId'] = dev_id
        df_tmp['latitude'] = lat
        df_tmp['longitude'] = lon
        
        all_weather_dfs.append(df_tmp)
        
    time.sleep(0.1) # Opóźnienie na rzecz publicznego darmowego źródła

if not all_weather_dfs:
    print("\n[BŁĄD] Nie pobrano nowych wartości.")
else:
    weather_df = pd.concat(all_weather_dfs, ignore_index=True)
    weather_df['time'] = pd.to_datetime(weather_df['time'])
    # Sprowadzamy kolumne do jednolitej nazwy zgodnej zazwyczaj z bazową ramką data_train/test:
    weather_df.rename(columns={'time': 'timedate'}, inplace=True)
    
    print(f"\nSUKCES! Gotowa połączona ramka liczy: {weather_df.shape[0]} rekordów historycznych o stałej częstotliwości godzinowej (1H).")
    display(weather_df.head(10))

Odpytywanie Punktów Open-Meteo:   0%|          | 0/600 [00:00<?, ?it/s]


SUKCES! Gotowa połączona ramka liczy: 1363584 rekordów historycznych o stałej częstotliwości godzinowej (1H).


,timedate,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,cloud_cover,deviceId,latitude,longitude
0,2024-10-01 00:00:00,5.8,92,0.0,996.4,7.1,0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,50.0,18.3
1,2024-10-01 01:00:00,4.7,94,0.0,996.0,6.1,2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,50.0,18.3
2,2024-10-01 02:00:00,3.4,97,0.0,995.8,8.2,0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,50.0,18.3
3,2024-10-01 03:00:00,3.2,99,0.0,995.5,6.9,2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,50.0,18.3
4,2024-10-01 04:00:00,3.0,100,0.0,995.2,5.9,32,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,50.0,18.3
5,2024-10-01 05:00:00,3.3,100,0.0,994.9,5.5,100,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,50.0,18.3
6,2024-10-01 06:00:00,4.4,97,0.0,994.9,7.8,95,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,50.0,18.3
7,2024-10-01 07:00:00,6.9,89,0.0,994.6,6.9,9,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,50.0,18.3
8,2024-10-01 08:00:00,10.0,77,0.0,994.4,7.8,4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,50.0,18.3
9,2024-10-01 09:00:00,12.6,64,0.0,993.9,7.5,8,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,50.0,18.3


## 3. Zapis surowych danych

In [5]:
if all_weather_dfs:
    OUTPUT_FILE = 'data/weather.csv'
    weather_df.to_csv(OUTPUT_FILE, index=False)
    print(f"Zapisano pełen, godzinny (`1H`) zrzut bez wstępnych agregacji pod ścieżką {OUTPUT_FILE}")

Zapisano pełen, godzinny (`1H`) zrzut bez wstępnych agregacji pod ścieżką data/weather.csv
